In [1]:
import numpy as np
import scipy as sp

In [2]:
def mc_price(S0, K, T, r, sigma, M):
    """
    Monte Carlo pricer for European Call/Put options
    """

    rng = np.random.default_rng()

    random_array = rng.standard_normal(M)

    S_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + random_array * sigma * np.sqrt(T) )

    discount = np.exp( - r * T )
    
    c_array = discount * np.maximum( S_array[:, np.newaxis] - K , 0 )
    p_array = discount * np.maximum( K - S_array[:, np.newaxis] , 0 )

    c_se = np.std( c_array, axis=0, ddof=1 ) / np.sqrt( M )
    p_se = np.std( p_array, axis=0, ddof=1 ) / np.sqrt( M )

    return c_array.mean( axis=0 ), p_array.mean( axis=0 ), c_se, p_se

In [12]:
S0 = 100.0
K = np.array([90, 100, 110]) 
T = 1.0
r = 0.05
sigma = 0.20
M = 10000

c, p, c_se, p_se = mc_price(S0, K, T, r, sigma, M)
print("Calls:", c, "SE:", c_se)
print("Puts:", p, "SE:", p_se)

Calls: [16.74554798 10.4541224   6.04378025] SE: [0.17357334 0.1471742  0.11635976]
Puts: [ 2.33351172  5.55438038 10.65633248] SE: [0.05432762 0.08729559 0.12030533]


In [10]:
def mc_price_antithetic(S0, K, T, r, sigma, M):
    """
    Monte Carlo pricer for European Call/Put options using antithetic variates
    """

    rng = np.random.default_rng()
    
    random_array = rng.standard_normal(M)
    neg_random_array = ( - 1 ) * random_array

    S1_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + random_array * sigma * np.sqrt(T) )
    S2_array = S1_array * np.exp( - 2 * sigma * neg_random_array * np.sqrt( T ) )

    discount = np.exp( - r * T )

    c1_array = discount * np.maximum( S1_array[:, np.newaxis] - K , 0 )
    c2_array = discount * np.maximum( S2_array[:, np.newaxis] - K , 0 )
    p1_array = discount * np.maximum( K - S1_array[:, np.newaxis] , 0 )
    p2_array = discount * np.maximum( K - S2_array[:, np.newaxis] , 0 )

    c_array = ( c1_array + c2_array ) / 2
    p_array = ( p1_array + p2_array ) / 2

    c_se = np.std( c_array, axis=0, ddof=1 ) / np.sqrt( M )
    p_se = np.std( p_array, axis=0, ddof=1 ) / np.sqrt( M )

    return c_array.mean( axis=0 ), p_array.mean( axis=0 ), c_se, p_se

In [13]:
S0 = 100.0
K = np.array([90, 100, 110]) 
T = 1.0
r = 0.05
sigma = 0.20
M = 10000

c, p, c_se, p_se = mc_price_antithetic(S0, K, T, r, sigma, M)
print("Calls:", c, "SE:", c_se)
print("Puts:", p, "SE:", p_se)

Calls: [30.32519972 24.48399033 19.90365592] SE: [0.42950998 0.40581943 0.37823242]
Puts: [ 6.49756623 10.16865108 15.10061092] SE: [0.10902353 0.14513154 0.18004559]
